In [10]:
# =========================
# Set up project path
# =========================

import sys
from pathlib import Path
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

# If this notebook is inside the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())

Current working directory: d:\2.DAP391m_Project\notebooks
Project root: d:\2.DAP391m_Project
Project root exists: True


In [11]:
# =========================
# Import Cleaned Dataset
# =========================

import pandas as pd
from src.config import CLEANED_DATA_PATH

print("Cleaned data path:", CLEANED_DATA_PATH)
print("File exists:", CLEANED_DATA_PATH.exists())

if not CLEANED_DATA_PATH.exists():
    raise FileNotFoundError(f"Cleaned dataset not found at: {CLEANED_DATA_PATH}")

df = pd.read_csv(CLEANED_DATA_PATH)

print("Cleaned dataset imported successfully!")
print("Shape:", df.shape)

display(df.head())

Cleaned data path: D:\2.DAP391m_Project\data\processed\dataset_cleaned.csv
File exists: True
Cleaned dataset imported successfully!
Shape: (288541, 9)


,user_id,timestamp,group,landing_page,converted,country,time_raw,elapsed_minutes,time_bucket
0,851104,11:48.6,control,old_page,0,US,11:48.6,11.810000,0-15_min
1,804228,01:45.2,control,old_page,0,US,01:45.2,1.753333,0-15_min
2,661590,55:06.2,treatment,new_page,0,US,55:06.2,55.103333,45-60_min
3,853541,28:03.1,treatment,new_page,0,US,28:03.1,28.051667,15-30_min
4,864975,52:26.2,control,old_page,1,US,52:26.2,52.436667,45-60_min


# Recalculate Original Conversion Rate and Uplift

Before applying bootstrap resampling, we first recalculate the original conversion rates of the old page and the new page using the cleaned dataset.

The original uplift is used as the baseline estimate. Bootstrap will then be used to estimate how much this uplift may vary if we repeatedly resample from the observed data.

The formulas are:

Conversion Rate = Number of Converted Users / Total Number of Users

Relative Uplift = ((CR_new - CR_old) / CR_old) × 100

In [12]:
# Separate users who viewed the old page
old_page_data = df[df["landing_page"] == "old_page"]

# Separate users who viewed the new page
new_page_data = df[df["landing_page"] == "new_page"]

# Calculate conversion rate for old_page
# Since converted is 0 or 1, the mean of converted is the conversion rate
old_page_conversion_rate = old_page_data["converted"].mean()

# Calculate conversion rate for new_page
new_page_conversion_rate = new_page_data["converted"].mean()

# Calculate relative uplift of new_page compared with old_page
# This tells us how much new_page increases or decreases conversion rate relative to old_page
original_uplift = (
    (new_page_conversion_rate - old_page_conversion_rate)
    / old_page_conversion_rate
) * 100

# Print the original conversion rates and uplift
print(f"Old page CR: {old_page_conversion_rate * 100:.2f}%")
print(f"New page CR: {new_page_conversion_rate * 100:.2f}%")
print(f"Original uplift: {original_uplift:.2f}%")

Old page CR: 12.03%
New page CR: 11.87%
Original uplift: -1.30%


## Bootstrap Resampling

Bootstrap is used to estimate the uncertainty of the relative uplift.

Instead of relying only on one observed uplift value, we repeatedly resample users from the old_page group and the new_page group with replacement. For each bootstrap sample, we recalculate the conversion rates and the relative uplift.

In this project, bootstrap is performed separately for old_page and new_page while preserving the original sample size of each group. This is appropriate for A/B testing because the old and new page groups are independent experimental groups.

We use 10,000 bootstrap iterations to obtain a stable empirical distribution of uplift values.

In [13]:
# Set random seed so that results can be reproduced
# This makes the random sampling process consistent every time the notebook is rerun
np.random.seed(42)

# Number of bootstrap iterations
# We use 10,000 iterations to obtain a stable bootstrap distribution
n_bootstrap = 10000

# Store the original sample size of each group
old_n = len(old_page_data)
new_n = len(new_page_data)

# Create an empty list to store uplift values from each bootstrap iteration
bootstrap_uplifts = []

# Run bootstrap resampling
for i in range(n_bootstrap):
    
    # Sample old_page users with replacement
    # The sample size is kept the same as the original old_page group
    old_sample = old_page_data.sample(
        n=old_n,
        replace=True
    )

    # Sample new_page users with replacement
    # The sample size is kept the same as the original new_page group
    new_sample = new_page_data.sample(
        n=new_n,
        replace=True
    )

    # Calculate conversion rate for the bootstrap old_page sample
    old_sample_cr = old_sample["converted"].mean()

    # Calculate conversion rate for the bootstrap new_page sample
    new_sample_cr = new_sample["converted"].mean()

    # Calculate relative uplift for this bootstrap sample
    sample_uplift = (
        (new_sample_cr - old_sample_cr)
        / old_sample_cr
    ) * 100

    # Save the uplift value
    bootstrap_uplifts.append(sample_uplift)

# Convert list to numpy array for easier numerical calculation
bootstrap_uplifts = np.array(bootstrap_uplifts)

## Calculate Bootstrap 95% Confidence Interval

After generating 10,000 bootstrap uplift values, we calculate the mean bootstrap uplift and the 95% confidence interval.

The 95% confidence interval is calculated using the percentile method:

- Lower bound = 2.5th percentile
- Upper bound = 97.5th percentile

If the confidence interval contains 0, it means that the uplift could be negative, zero, or slightly positive. In that case, the observed uplift is not statistically robust.

In [14]:
# Calculate the mean of all bootstrap uplift values
mean_bootstrap_uplift = bootstrap_uplifts.mean()

# Calculate the lower bound of the 95% confidence interval
# 2.5th percentile leaves 2.5% of values below this point
ci_lower = np.percentile(bootstrap_uplifts, 2.5)

# Calculate the upper bound of the 95% confidence interval
# 97.5th percentile leaves 2.5% of values above this point
ci_upper = np.percentile(bootstrap_uplifts, 97.5)

# Print the bootstrap result
print(f"Mean bootstrap uplift: {mean_bootstrap_uplift:.2f}%")
print(f"95% Bootstrap CI: [{ci_lower:.2f}%, {ci_upper:.2f}%]")

# Check whether the confidence interval contains zero
ci_contains_zero = ci_lower <= 0 <= ci_upper

print("CI contains zero:", ci_contains_zero)

Mean bootstrap uplift: -1.29%
95% Bootstrap CI: [-3.24%, 0.71%]
CI contains zero: True


## Inspect Bootstrap Results

Before saving the bootstrap results, we inspect the generated bootstrap values.

The detailed bootstrap result table contains one row for each bootstrap iteration. The summary table contains the key statistics that will be used in the report:

- Original uplift
- Mean bootstrap uplift
- Lower bound of 95% CI
- Upper bound of 95% CI
- Number of bootstrap iterations
- Whether the CI contains 0

In [15]:
# Create a table containing all bootstrap uplift values
# Each row represents one bootstrap iteration
bootstrap_results = pd.DataFrame({
    "bootstrap_iteration": range(1, n_bootstrap + 1),
    "bootstrap_uplift": bootstrap_uplifts
})

# Display the first few rows
bootstrap_results.head()


,bootstrap_iteration,bootstrap_uplift
0,1,-1.792453
1,2,-1.121413
2,3,-2.889574
3,4,-0.509104
4,5,-1.007386


In [16]:
# Inspect the distribution of bootstrap uplift values
bootstrap_results["bootstrap_uplift"].describe()

count    10000.000000
mean        -1.286764
std          1.006622
min         -5.158316
25%         -1.960655
50%         -1.297156
75%         -0.609784
max          2.367100
Name: bootstrap_uplift, dtype: float64

In [17]:
# Create a summary table for the bootstrap analysis
bootstrap_summary = pd.DataFrame({
    "original_uplift": [original_uplift],
    "mean_bootstrap_uplift": [mean_bootstrap_uplift],
    "ci_lower": [ci_lower],
    "ci_upper": [ci_upper],
    "n_bootstrap": [n_bootstrap],
    "ci_contains_zero": [ci_contains_zero]
})

# Display the summary table
bootstrap_summary

,original_uplift,mean_bootstrap_uplift,ci_lower,ci_upper,n_bootstrap,ci_contains_zero
0,-1.300171,-1.286764,-3.243567,0.711739,10000,True


## Save Bootstrap Results

Two result files are saved:

1. `bootstrap_results.csv`  
   This file contains all 10,000 bootstrap uplift values.

2. `bootstrap_summary.csv`  
   This file contains the main bootstrap summary, including the original uplift, mean bootstrap uplift, 95% confidence interval, and whether the confidence interval contains 0.

These files will be used later for visualization and reporting.

In [19]:
# Define the results directory
# If the folder does not exist, create it automatically
results_dir = Path("../data/results")
results_dir.mkdir(parents=True, exist_ok=True)

# Save all bootstrap uplift values
bootstrap_results.to_csv(
    results_dir / "bootstrap_results.csv",
    index=False
)

# Save bootstrap summary
bootstrap_summary.to_csv(
    results_dir / "bootstrap_summary.csv",
    index=False
)

# Check whether the files were saved successfully
print((results_dir / "bootstrap_results.csv").exists())
print((results_dir / "bootstrap_summary.csv").exists())


True
True


## Summary

In this notebook, bootstrap resampling was used to estimate the uncertainty of the overall relative uplift between the new page and the old page.

The analysis followed these steps:

1. Recalculate original conversion rates and relative uplift.
2. Resample old_page and new_page users separately with replacement.
3. Repeat the resampling process 10,000 times.
4. Calculate relative uplift for each bootstrap sample.
5. Estimate the 95% confidence interval using the percentile method.
6. Save detailed bootstrap results and summary tables.

The bootstrap result shows that the confidence interval contains 0. Therefore, the new page does not show statistically robust evidence of improving conversion rate.